# P03: Applications and advanced tips

Lists are the natural home for a series of measurements -- a column of reaction
times, a run of spike counts. Here we slice out time windows, compute summary
statistics by hand (to see the mechanics), and meet the single most common
list bug in data analysis: accidental aliasing.

## Reaction times as a list

In [1]:
rts = [412, 520, 380, 660, 290, 505]   # reaction times in ms

# slicing pulls out a sub-sequence -- great for splitting into blocks/windows
first_block = rts[:3]      # trials 0, 1, 2
last_block = rts[3:]       # trials 3, 4, 5
print(first_block, last_block)

# the mean, done by hand, just to see what numpy will later do for us
mean_rt = sum(rts) / len(rts)
print(f"mean RT = {mean_rt:.1f} ms")

[412, 520, 380] [660, 290, 505]
mean RT = 461.2 ms


## A median by hand (watch the even-length case)

In [2]:
s = sorted(rts)
n = len(s)
# with an even number of points the median is the average of the middle two
median_rt = s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2
print(median_rt)

458.5


:::{admonition} Double check: the easy-to-miss even-length median
:class: warning
A quick implementation that just returns `sorted(rts)[len(rts) // 2]` is correct
for odd-length lists and silently wrong for even-length ones (it returns the
upper-middle value, not the average of the two middle values). It runs fine on
every input and only biases your statistic. Prefer `numpy.median`/`statistics.median`,
and if you must write it yourself, handle both cases as above.
:::

## Aliasing: the bug that quietly edits your raw data

In [3]:
# assignment does NOT copy a list -- both names point to the SAME list in memory
raw = [412, 520, 380]
cleaned = raw                 # this is an alias, not a copy!
cleaned[0] = 999
print("raw is now:", raw)     # raw changed too -- almost never what you want

raw is now: [999, 520, 380]


In [4]:
# make an independent copy instead
raw = [412, 520, 380]
cleaned = raw.copy()          # or raw[:] or list(raw)
cleaned[0] = 999
print("raw:", raw, " cleaned:", cleaned)

raw: [412, 520, 380]  cleaned: [999, 520, 380]


:::{admonition} Double check: AI 'cleaning' code can mutate your raw data in place
:class: warning
If you ask an assistant to "clean the reaction times," a common pattern is
`cleaned = raw` followed by edits to `cleaned`. Because lists are references, this
also rewrites `raw` -- your original data is gone, with no error and no warning.
Always copy before you mutate, and keep an untouched copy of raw data. (`numpy`
arrays and `pandas` objects have the very same view-vs-copy hazard, which we
revisit in P07 and P09.)
:::